# Memory coalescing on a T4: offset, stride and the naive transpose

**Runtime → Change runtime type → T4 GPU**, then run all cells (about a minute).

You will measure:
1. **Offset sweep**: a warp reads 32 consecutive floats starting `s` elements past an aligned address (s = 0..32). The coalescing card's lab predicts 4 sectors when s is a multiple of 8 and 5 sectors otherwise (80% useful bytes).
2. **Stride sweep**: lane k reads element `k * s` (s = 1..32). The lab predicts 50% useful bytes at stride 2 and 12.5% from stride 8 on.
3. **copy vs transposeNaive** on a 1024 × 1024 float matrix: same reads, but the naive transpose writes with a 4096-byte stride.

Every number is printed as GB/s of *useful* bytes and as a % of the T4's 320 GB/s peak. Colab's T4 clocks vary, so expect run-to-run noise of a few percent.

Card: https://seyonv.github.io/explainers/perf-3-kernels/coalescing.html · kernels adapted from Mark Harris's NVIDIA blog posts (links at the end).

Not yet run by the author (no NVIDIA GPU). The reference numbers at the end come from the sources' GPUs; your T4 run is the exercise.

In [ ]:
!nvidia-smi

## 1–2. Offset and stride kernels
Each launch touches N = 8,388,608 floats (32 MiB), is warmed up once, and is averaged over 20 launches timed with `cudaEvent`. The blog used 4 MB, which is too short to time reliably on a modern GPU.

In [ ]:
%%writefile coalescing.cu
// Offset and stride kernels from Harris, "How to Access Global Memory Efficiently
// in CUDA C/C++ Kernels" (NVIDIA blog, 2012), adapted: float only, bigger array,
// averaged over REPS launches, every CUDA call checked, results verified.
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CUDA_CHECK(call) do { cudaError_t e_ = (call); if (e_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); \
  exit(1); } } while (0)

__global__ void offset(float* a, int s)
{
  int i = blockDim.x * blockIdx.x + threadIdx.x + s;
  a[i] = a[i] + 1;
}

__global__ void stride(float* a, int s)
{
  int i = (blockDim.x * blockIdx.x + threadIdx.x) * s;
  a[i] = a[i] + 1;
}

const int BLOCK = 256;
const int N = 8 * 1024 * 1024;          // 8,388,608 floats = 32 MiB touched per launch
const int REPS = 20;
const double T4_PEAK = 320.0;           // GB/s, NVIDIA T4 datasheet

// Effective bandwidth counts only the bytes the threads use: N floats read + N written.
double run(bool isStride, float* d_a, int s, cudaEvent_t t0, cudaEvent_t t1)
{
  int grid = N / BLOCK;
  if (isStride) stride<<<grid, BLOCK>>>(d_a, s); else offset<<<grid, BLOCK>>>(d_a, s);  // warm-up
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaEventRecord(t0));
  for (int r = 0; r < REPS; r++) {
    if (isStride) stride<<<grid, BLOCK>>>(d_a, s); else offset<<<grid, BLOCK>>>(d_a, s);
  }
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaEventRecord(t1));
  CUDA_CHECK(cudaEventSynchronize(t1));
  float ms = 0;
  CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));
  double sec = ms / 1e3 / REPS;
  return 2.0 * N * sizeof(float) / sec / 1e9;
}

// Run one launch on zeroed memory and check that exactly the expected elements became 1.
bool verify(bool isStride, float* d_a, int s)
{
  size_t len = isStride ? (size_t)N * s : (size_t)N + s;
  CUDA_CHECK(cudaMemset(d_a, 0, len * sizeof(float)));
  if (isStride) stride<<<N / BLOCK, BLOCK>>>(d_a, s); else offset<<<N / BLOCK, BLOCK>>>(d_a, s);
  CUDA_CHECK(cudaGetLastError());
  std::vector<float> h(len);
  CUDA_CHECK(cudaMemcpy(h.data(), d_a, len * sizeof(float), cudaMemcpyDeviceToHost));
  for (size_t j = 0; j < len; j++) {
    bool touched = isStride ? (j % s == 0) : (j >= (size_t)s);
    if (h[j] != (touched ? 1.0f : 0.0f)) { printf("mismatch at %zu\n", j); return false; }
  }
  return true;
}

int main()
{
  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("Device: %s, %d SMs\n", prop.name, prop.multiProcessorCount);

  float* d_a;
  CUDA_CHECK(cudaMalloc(&d_a, (size_t)N * 33 * sizeof(float)));  // room for stride 32
  CUDA_CHECK(cudaMemset(d_a, 0, (size_t)N * 33 * sizeof(float)));
  cudaEvent_t t0, t1;
  CUDA_CHECK(cudaEventCreate(&t0));
  CUDA_CHECK(cudaEventCreate(&t1));

  printf("\nOffset sweep (stride 1)\n%6s %10s %10s\n", "offset", "GB/s", "% of 320");
  for (int s = 0; s <= 32; s++) {
    double gbs = run(false, d_a, s, t0, t1);
    printf("%6d %10.1f %9.1f%%\n", s, gbs, 100.0 * gbs / T4_PEAK);
  }
  printf("\nStride sweep (offset 0)\n%6s %10s %10s\n", "stride", "GB/s", "% of 320");
  for (int s = 1; s <= 32; s++) {
    double gbs = run(true, d_a, s, t0, t1);
    printf("%6d %10.1f %9.1f%%\n", s, gbs, 100.0 * gbs / T4_PEAK);
  }

  bool ok = verify(false, d_a, 1) && verify(true, d_a, 2) && verify(true, d_a, 32);
  printf("\nVerification (offset 1, stride 2, stride 32): %s\n", ok ? "PASSED" : "FAILED");

  CUDA_CHECK(cudaEventDestroy(t0));
  CUDA_CHECK(cudaEventDestroy(t1));
  CUDA_CHECK(cudaFree(d_a));
  return ok ? 0 : 1;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o coalescing coalescing.cu && ./coalescing

## 3. copy vs transposeNaive (1024 × 1024 floats)
Same launch shape as the blog: 32 × 8 threads per 32 × 32 tile, 100 timed repetitions, effective bandwidth = 2 × matrix bytes ÷ time. Results are checked against a CPU transpose.

In [ ]:
%%writefile transpose_naive.cu
// copy vs transposeNaive from Harris, "An Efficient Matrix Transpose in CUDA C/C++"
// (NVIDIA blog), same sizes: 1024 x 1024 floats, 32 x 8 threads per 32 x 32 tile.
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CUDA_CHECK(call) do { cudaError_t e_ = (call); if (e_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); \
  exit(1); } } while (0)

const int TILE_DIM = 32;
const int BLOCK_ROWS = 8;
const int NUM_REPS = 100;
const double T4_PEAK = 320.0;

__global__ void copy(float* odata, const float* idata)
{
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[(y + j) * width + x] = idata[(y + j) * width + x];
}

// Reads are coalesced; writes jump by `width` floats (4096 B) between neighbouring threads.
__global__ void transposeNaive(float* odata, const float* idata)
{
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[x * width + (y + j)] = idata[(y + j) * width + x];
}

int main()
{
  const int nx = 1024, ny = 1024, n = nx * ny;
  const size_t bytes = (size_t)n * sizeof(float);
  dim3 grid(nx / TILE_DIM, ny / TILE_DIM), block(TILE_DIM, BLOCK_ROWS);

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("Device: %s, ECC %s\n", prop.name, prop.ECCEnabled ? "on" : "off");

  std::vector<float> h_in(n), h_out(n), ref_copy(n), ref_t(n);
  for (int i = 0; i < n; i++) h_in[i] = (float)i;
  for (int y = 0; y < ny; y++)
    for (int x = 0; x < nx; x++) {
      ref_copy[y * nx + x] = h_in[y * nx + x];
      ref_t[x * ny + y] = h_in[y * nx + x];
    }

  float *d_in, *d_out;
  CUDA_CHECK(cudaMalloc(&d_in, bytes));
  CUDA_CHECK(cudaMalloc(&d_out, bytes));
  CUDA_CHECK(cudaMemcpy(d_in, h_in.data(), bytes, cudaMemcpyHostToDevice));
  cudaEvent_t t0, t1;
  CUDA_CHECK(cudaEventCreate(&t0));
  CUDA_CHECK(cudaEventCreate(&t1));

  printf("%-16s %10s %10s %8s\n", "kernel", "GB/s", "% of 320", "check");
  for (int k = 0; k < 2; k++) {
    const char* name = k == 0 ? "copy" : "transposeNaive";
    const std::vector<float>& ref = k == 0 ? ref_copy : ref_t;
    CUDA_CHECK(cudaMemset(d_out, 0, bytes));
    if (k == 0) copy<<<grid, block>>>(d_out, d_in); else transposeNaive<<<grid, block>>>(d_out, d_in);  // warm-up
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r = 0; r < NUM_REPS; r++) {
      if (k == 0) copy<<<grid, block>>>(d_out, d_in); else transposeNaive<<<grid, block>>>(d_out, d_in);
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));
    float ms = 0;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));
    CUDA_CHECK(cudaMemcpy(h_out.data(), d_out, bytes, cudaMemcpyDeviceToHost));
    bool ok = true;
    for (int i = 0; i < n; i++) if (h_out[i] != ref[i]) { ok = false; break; }
    double gbs = 2.0 * bytes * NUM_REPS / (ms / 1e3) / 1e9;   // same metric as the blog
    printf("%-16s %10.1f %9.1f%% %8s\n", name, gbs, 100.0 * gbs / T4_PEAK, ok ? "PASSED" : "FAILED");
  }

  CUDA_CHECK(cudaEventDestroy(t0));
  CUDA_CHECK(cudaEventDestroy(t1));
  CUDA_CHECK(cudaFree(d_in));
  CUDA_CHECK(cudaFree(d_out));
  return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o transpose_naive transpose_naive.cu && ./transpose_naive

## Reference numbers (from the sources, not from a T4)

| Source | GPU | What it reports |
|---|---|---|
| Harris, *How to Access Global Memory Efficiently* | Tesla C870 (CC 1.0) | misaligned or non-unit-stride access: "the familiar 1/8 bandwidth" |
| same | Tesla C1060 (CC 1.3) | misaligned access "less problematic"; stride gives "a smooth bandwidth curve" |
| same | Tesla C2050 (CC 2.0) | "negligible effect of alignment"; large strides poor "regardless of architecture" |
| Harris, *An Efficient Matrix Transpose* (ECC on) | Tesla M2050 | copy **105.2** GB/s, transposeNaive **18.8** GB/s |
| same | Tesla K20c | copy **136.0** GB/s, transposeNaive **55.3** GB/s |
| CUDA C++ Best Practices Guide §10.2.1.3 | Tesla V100 | aligned ≈ 790 GB/s; misaligned expected ≈ 4/5, achieved ≈ 9/10 (neighbouring warps reuse cache lines) |

The blog's offset/stride results are plots only, so no exact values are quoted here.

Questions to answer from your run:
- Does the offset sweep dip by ~20% at offsets that are not multiples of 8, or is it nearly flat (caches hiding it, like the C2050 and V100)?
- Where does the stride curve flatten? The sector model says it can't get worse after stride 8.
- Is your naive/copy ratio closer to the M2050's 5.6× or the K20c's 2.5×?

## Try this
1. Change `float` to `double` in both kernels: the offset penalty moves to multiples of 4 elements (32 B ÷ 8 B).
2. If `ncu` works in your runtime: `!ncu --metrics l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum ./coalescing` and divide sectors by requests (NVIDIA's 2025 blog: 4 coalesced, 32 uncoalesced).
3. Fix the transpose with a shared-memory tile: see the next card, shared-memory-transpose.html, and its notebook.

Sources: https://developer.nvidia.com/blog/how-access-global-memory-efficiently-cuda-c-kernels/ · https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/ · https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/ · https://developer.nvidia.com/blog/unlock-gpu-performance-global-memory-access-in-cuda/